# Architectural Deep-Dive: 1st Place Solution

Welcome to the deep-dive notebook for my 1st place solution. Rather than a standard trial-and-error approach, this solution was built on deliberate architectural and design decisions targeted at specific dataset challenges.

## 📊 Performance Summary
| Architecture / Setup | Key Design Choice | F1 Score |
| --- | --- | --- |
| **TabPFN (Finetuned)** | Focal Loss + Deterministic PR Curve Optimization | **0.82** |
| **AutoGluon Ensemble** | 20x Bagging + 3-Layer Stacking | **0.80** |
| **TabPFN Baseline** | GPU-Accelerated Expected F1 Maximization | **0.79** |

## 🛠 Setup
First, let's load our environment and custom modules.

In [ ]:
import sys
import os

# Add src to path so we can import our modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import our custom modules
from train_autogluon import train_autogluon
from train_tabpfn import train_tabpfn
from monte_carlo import run_simulation_gpu, plot_threshold_analysis
from train_finetune import run_finetuning

%matplotlib inline

## 🔍 1. Defining the Problems (EDA)

Before making architectural decisions, we need to understand the constraints of the data.

### Challenge 1: Extreme Class Imbalance
![Class Imbalance](../resources/class_imbalance.png)
The dataset exhibits a severe class imbalance. Standard Cross-Entropy loss functions will inevitably bias the network towards the majority class, destroying recall on the minority class.

### Challenge 2: Complex Feature Interactions
![Correlation Matrix](../resources/correlation_matrix.png)
The numerical features share dense, non-linear correlations. Tree-based models require massive depth to capture these interactions, leading to overfitting.

## 2. Decision 1: Transformers over Trees (Score: 0.79)

To solve Challenge 2, I selected **TabPFN**, a transformer designed for tabular data. By using self-attention mechanisms, TabPFN evaluates all feature interactions simultaneously, naturally modeling the dense correlations.

### Decision 2: GPU-Accelerated Monte Carlo Thresholding
Classification models output probabilities, but we are evaluated on F1 Score. The default `0.5` threshold fails on imbalanced data. To solve this, I built a GPU-accelerated Monte Carlo simulation. This allowed me to test thousands of thresholds across millions of synthetic probability distributions to find the most mathematically robust decision boundary without exceeding compute timeouts.

In [ ]:
# model, tabpfn_probs, ids = train_tabpfn('../data/dataset-train-vf (2).csv', '../data/dataset-test-vf.csv', max_time=600)
# thresholds, mean_f1, std_f1 = run_simulation_gpu(tabpfn_probs, n_sims=1000000, steps=1000, batch_size=5000)
# best_idx = mean_f1.argmax()
# best_thresh = thresholds[best_idx]

## 3. Decision 3: Variance Reduction via Ensembling (Score: 0.80)
Deep learning models inherently suffer from higher variance. To stabilize the predictions, I integrated an AutoGluon stacked ensemble. By employing **20x Bagging** and **3-layer Stacking**, the ensemble aggregated predictions across diverse model types, effectively crushing model variance and ensuring a highly stable baseline.

In [ ]:
# predictor, ag_probs, ids = train_autogluon('../data/dataset-train-vf (2).csv', '../data/dataset-test-vf.csv')

## 4. The Final Architecture: Finetuned TabPFN (Score: 0.82) 🏆
This pipeline combines all the insights into a single, highly optimized script.

### Decision 4: Loss Function Engineering (Focal Loss)
To solve Challenge 1 (Extreme Imbalance), I implemented a custom **Focal Loss** (`alpha=0.75`, `gamma=2.0`). This dynamically scales the cross-entropy loss, mathematically down-weighting easy examples and forcing the gradient updates to focus entirely on the hard-to-classify minority cases.

### Decision 5: Deterministic PR-Curve Maximization
I paired the Focal Loss with a deterministic Precision-Recall curve extraction. This algorithm mathematically pinpoints the exact threshold that yields the absolute maximum F1 score on the validation set.


In [ ]:
# Run the highly-optimized finetuning script
# final_model, preprocessor, optimal_threshold = run_finetuning(train_path='../data/dataset-train-vf (2).csv')

## Conclusion
By treating the competition as an engineering design problem—identifying specific dataset constraints and systematically building mathematical and architectural solutions to address them—I was able to secure a stable and robust 1st place finish.